# Coverage Context Features

The first WR model used nearest-defender geometry at the snap. That is a useful baseline, but it is too crude for zone coverage because the nearest defender is not always the defender who matters most.

This notebook evaluates a richer defensive-context feature set:

- nearest, second-nearest, and third-nearest defender distance at the snap
- defenders within 3, 5, and 10 yards
- nearest DB and nearest LB distance
- nearest-defender depth and width leverage
- defenders ahead of the receiver within 10 yards
- close defenders inside/outside the receiver

The question is whether this richer context improves WR prediction and stability.

In [1]:
from pathlib import Path
import csv
import statistics
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from separation_over_expected.reports import pearson_correlation, read_csv_rows

ROUTE_TABLE = PROJECT_ROOT / "data" / "processed" / "route_level_snap_to_release.csv"
WR_METRICS = PROJECT_ROOT / "data" / "processed" / "position_baselines" / "baseline_metrics_wr.csv"
STABILITY = PROJECT_ROOT / "data" / "processed" / "stability" / "wr_split_half_stability.csv"

PROJECT_ROOT

PosixPath('/Users/moggo/Projects/NFL/Separation_Over_Expected')

## New Columns

The rebuilt route table has additional defender-context columns. These are now included in the ridge context model.

In [2]:
with ROUTE_TABLE.open(newline="") as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames

[column for column in columns if "defender" in column and column.endswith("_snap")]

['nearest_defender_dx_snap', 'nearest_defender_dy_snap', 'defender_1_dist_snap', 'defender_2_dist_snap', 'defender_3_dist_snap', 'defenders_within_3_snap', 'defenders_within_5_snap', 'defenders_within_10_snap', 'nearest_defender_depth_leverage_snap', 'nearest_defender_width_leverage_snap', 'defender_depth_density_0_10_snap', 'defender_inside_count_5_snap', 'defender_outside_count_5_snap']

## WR Test Metrics

The original WR ridge model had:

```text
test R2   = 0.503
test RMSE = 1.848
test MAE  = 1.393
```

The richer coverage-context model keeps essentially the same test R2 and RMSE, with a small MAE improvement.

In [3]:
metrics = read_csv_rows(WR_METRICS)
[row for row in metrics if row["split"] == "test"]

[{'model': 'global_mean', 'split': 'test', 'rows': '2474', 'mae': '2.012', 'rmse': '2.626', 'r2': '-0.004', 'bias': '-0.163', 'mean_actual': '-2.480', 'mean_predicted': '-2.317', 'n': '2474'}, {'model': 'smoothed_group_mean', 'split': 'test', 'rows': '2474', 'mae': '2.003', 'rmse': '2.619', 'r2': '0.002', 'bias': '-0.167', 'mean_actual': '-2.480', 'mean_predicted': '-2.313', 'n': '2474'}, {'model': 'ridge_context', 'split': 'test', 'rows': '2474', 'mae': '1.388', 'rmse': '1.848', 'r2': '0.503', 'bias': '-0.048', 'mean_actual': '-2.480', 'mean_predicted': '-2.432', 'n': '2474'}]

## Split-Half Stability

The original WR split-half stability correlation was 0.420. After adding these coverage-context features, the correlation is slightly lower.

In [4]:
stability = read_csv_rows(STABILITY)
early = [float(row["early_mean_soe"]) for row in stability]
late = [float(row["late_mean_soe"]) for row in stability]
correlation = pearson_correlation(early, late)

print(f"Qualified WRs: {len(stability)}")
print(f"Early/late Pearson correlation: {correlation:.3f}")

Qualified WRs: 107
Early/late Pearson correlation: 0.406


In [5]:
stability[:15]

[{'nflId': '53478', 'displayName': 'Rondale Moore', 'officialPosition': 'WR', 'early_routes': '42', 'late_routes': '55', 'early_mean_soe': '2.224', 'late_mean_soe': '1.664', 'late_minus_early': '-0.560'}, {'nflId': '43010', 'displayName': 'Adam Humphries', 'officialPosition': 'WR', 'early_routes': '92', 'late_routes': '102', 'early_mean_soe': '0.674', 'late_mean_soe': '-0.284', 'late_minus_early': '-0.958'}, {'nflId': '46256', 'displayName': 'Ray-Ray McCloud', 'officialPosition': 'WR', 'early_routes': '36', 'late_routes': '41', 'early_mean_soe': '0.653', 'late_mean_soe': '-0.268', 'late_minus_early': '-0.921'}, {'nflId': '52622', 'displayName': 'Freddie Swain', 'officialPosition': 'WR', 'early_routes': '65', 'late_routes': '82', 'early_mean_soe': '0.652', 'late_mean_soe': '0.373', 'late_minus_early': '-0.279'}, {'nflId': '42448', 'displayName': 'Jamison Crowder', 'officialPosition': 'WR', 'early_routes': '20', 'late_routes': '92', 'early_mean_soe': '0.552', 'late_mean_soe': '0.290', 'l

## Takeaway

This feature set did not materially improve the WR model. Test error is almost unchanged, and split-half stability moved from 0.420 to about 0.406.

That does not mean defensive context is unimportant. It means these simple density/leverage features are not enough to solve the defender-assignment problem. The next improvement should probably focus on one of two directions:

1. Add route-shape or route-phase information so defenders are interpreted relative to the receiver's actual path.
2. Build a better coverage-responsibility proxy, such as weighted defender distance over time rather than snap-only density.